IMPORTING ALL THE NECESSARY LIBRARIES

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

IMPORTING THE INDIAN AIR QUALITY DATASETS FOR THE DIFFERENT CITIES

In [2]:
import glob
import os

# 1. Unzip the file
!unzip -q raw_data.zip

# 2. Find all CSV files INSIDE the folder
all_files = glob.glob("raw_data/*.csv")
print(f"Found {len(all_files)} files.")

# 3. Load each CSV
dataframes = []
for filename in all_files:
    df = pd.read_csv(filename)
    dataframes.append(df)

# 4. Merge all datasets
full_df = pd.concat(dataframes, ignore_index=True)
print(f"Merged Data Shape: {full_df.shape}")

# 5. Save merged dataset
full_df.to_csv("merged_data.csv", index=False)
print("merged_data.csv saved successfully!")

Found 26 files.
Merged Data Shape: (29531, 16)
merged_data.csv saved successfully!


PREPARE THE DATASET FOR CLEANING AND PREPROCESSING

In [3]:
df = pd.read_csv("/content/merged_data.csv", encoding='latin1')

print("Dataset Ready for Cleaning!")

Dataset Ready for Cleaning!


TESTING THE DATASET

In [5]:
df.head()

,City,Date,PM2.5,PM10,NO,NO2,NOx,NH3,CO,SO2,O3,Benzene,Toluene,Xylene,AQI,AQI_Bucket
0,Aizawl,11/03/2020,32.69,47.91,6.99,2.85,11.93,26.64,0.60,4.53,4.48,0.03,0.30,NaN,NaN,NaN
1,Aizawl,12/03/2020,31.21,38.66,7.20,1.27,10.65,25.63,0.56,4.22,2.81,0.01,0.08,NaN,52.0,Satisfactory
2,Aizawl,13/03/2020,38.39,46.68,7.19,0.91,10.37,29.16,0.57,4.46,0.18,0.00,0.00,NaN,60.0,Satisfactory
3,Aizawl,14/03/2020,43.23,50.83,7.14,1.07,10.48,28.95,0.57,4.53,0.41,0.00,0.00,NaN,62.0,Satisfactory
4,Aizawl,15/03/2020,33.82,41.03,7.09,0.36,9.73,28.41,0.48,4.63,0.30,0.00,0.00,NaN,70.0,Satisfactory


In [6]:
print("Shape of dataset:", df.shape)

Shape of dataset: (29531, 16)


CLEANING THE DATASET;

STEP 1; CHECKING FOR COLUMN INFORMATION

In [7]:
print("\nColumn information:")
df.info()


Column information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 29531 entries, 0 to 29530
Data columns (total 16 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   City        29531 non-null  object 
 1   Date        29531 non-null  object 
 2   PM2.5       24933 non-null  float64
 3   PM10        18391 non-null  float64
 4   NO          25949 non-null  float64
 5   NO2         25946 non-null  float64
 6   NOx         25346 non-null  float64
 7   NH3         19203 non-null  float64
 8   CO          27472 non-null  float64
 9   SO2         25677 non-null  float64
 10  O3          25509 non-null  float64
 11  Benzene     23908 non-null  float64
 12  Toluene     21490 non-null  float64
 13  Xylene      11422 non-null  float64
 14  AQI         24850 non-null  float64
 15  AQI_Bucket  24850 non-null  object 
dtypes: float64(13), object(3)
memory usage: 3.6+ MB


STEP 2; CHECKING FOR MISSING DATA WITHIN EACH COLUMN

In [8]:
print("\nMissing values")
print(df.isna().sum())


Missing values
City              0
Date              0
PM2.5          4598
PM10          11140
NO             3582
NO2            3585
NOx            4185
NH3           10328
CO             2059
SO2            3854
O3             4022
Benzene        5623
Toluene        8041
Xylene        18109
AQI            4681
AQI_Bucket     4681
dtype: int64


STEP 3; REMOVE COLUMNS WITH OVER 50% OF DATA MISSING

In [9]:
# Drop columns with too many missing values (>50%)
missing_percent = df.isna().mean() * 100
cols_to_drop = missing_percent[missing_percent > 50].index
df.drop(columns=cols_to_drop, inplace=True)

print(f"\nDropped columns with >50% missing values: {list(cols_to_drop)}")


Dropped columns with >50% missing values: ['Xylene']


STEP 4; REPLACE MISSING POLLUTION DATA USING MEDIAN

In [10]:
# Update pollution column list
pollution_cols = [
    'PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3',
    'CO', 'SO2', 'O3', 'Benzene', 'Toluene'
]

# 2. City-wise median imputation for remaining pollutants
df[pollution_cols] = (
    df
    .groupby('City')[pollution_cols]
    .transform(lambda x: x.fillna(x.median()))
)

# 3. Final safety check: fill any remaining missing values using global median
df[pollution_cols] = df[pollution_cols].fillna(
    df[pollution_cols].median()
)

# 4. Verify missing values are handled
print(df[pollution_cols].isnull().sum())

PM2.5      0
PM10       0
NO         0
NO2        0
NOx        0
NH3        0
CO         0
SO2        0
O3         0
Benzene    0
Toluene    0
dtype: int64


STEP 5; GIVEN THAT AQI (AIR QUALITY INDEX) IS THE TARGET VARIABLE, MISSING DATA WILL BE REMOVED FOR THAT ROW.

In [11]:
# Drop rows where AQI is missing
df = df.dropna(subset=['AQI'])

STEP 6; TESTING THE DATASET TO OBSERVE ANYMORE MISSING DATA

In [12]:
print("\nMissing values")
print(df.isna().sum())


Missing values
City          0
Date          0
PM2.5         0
PM10          0
NO            0
NO2           0
NOx           0
NH3           0
CO            0
SO2           0
O3            0
Benzene       0
Toluene       0
AQI           0
AQI_Bucket    0
dtype: int64


STEP 7; ADJUSTING THE DATA COLUMN INTO YEARS, QUARTERLY AND MONTHS

BINNING THE YEARS

In [13]:
# Convert to datetime, invalid dates become NaT
df['Date'] = pd.to_datetime(df['Date'], errors='coerce', format='%d/%m/%Y')

# Check for invalid dates
print(df[df['Date'].isna()])

# Define year bins from 2006 to 2024
year_bins = list(range(2014, 2021))  # edges
year_labels = list(range(2015, 2021))  # labels

# Bin the years
df['Year_bin'] = pd.cut(df['Date'].dt.year,
                                  bins=year_bins,
                                  labels=year_labels,
                                  include_lowest=True)

# Check
print(df[['Date', 'Year_bin']].head())

Empty DataFrame
Columns: [City, Date, PM2.5, PM10, NO, NO2, NOx, NH3, CO, SO2, O3, Benzene, Toluene, AQI, AQI_Bucket]
Index: []
        Date Year_bin
1 2020-03-12     2020
2 2020-03-13     2020
3 2020-03-14     2020
4 2020-03-15     2020
5 2020-03-16     2020


BINNING FOR QUARTERLY PERIODS

In [14]:
# Convert to datetime, invalid dates become NaT
df['Date'] = pd.to_datetime(df['Date'], errors='coerce', format='%d/%m/%Y')

# Check for invalid dates
print(df[df['Date'].isna()])

# Define month bins (e.g., 1-3, 4-6, 7-9, 10-12)
month_bins = [0, 3, 6, 9, 12]  # edges
month_labels = ['Jan-Mar', 'Apr-Jun', 'Jul-Sep', 'Oct-Dec']  # labels

# Bin the months
df['Quarterly_bin'] = pd.cut(df['Date'].dt.month,
                         bins=month_bins,
                         labels=month_labels,
                         include_lowest=True)

# Check
print(df[['Date', 'Quarterly_bin']].head())

Empty DataFrame
Columns: [City, Date, PM2.5, PM10, NO, NO2, NOx, NH3, CO, SO2, O3, Benzene, Toluene, AQI, AQI_Bucket, Year_bin]
Index: []
        Date Quarterly_bin
1 2020-03-12       Jan-Mar
2 2020-03-13       Jan-Mar
3 2020-03-14       Jan-Mar
4 2020-03-15       Jan-Mar
5 2020-03-16       Jan-Mar


BINNING THE MONTHS

In [15]:
# Convert to datetime, invalid dates become NaT
df['Date'] = pd.to_datetime(df['Date'], errors='coerce', format='%d/%m/%Y')

# Check for invalid dates
print(df[df['Date'].isna()])

# Define month bins (1 to 12)
month_bins = list(range(0, 13))  # edges: 0,1,2,...,12
month_labels = ['January', 'February', 'March', 'April', 'May', 'June',
                'July', 'August', 'September', 'October', 'November', 'December']

# Bin the months
df['Month_bin'] = pd.cut(df['Date'].dt.month,
                         bins=month_bins,
                         labels=month_labels,
                         include_lowest=True)

# Check
print(df[['Date', 'Month_bin']].head())

Empty DataFrame
Columns: [City, Date, PM2.5, PM10, NO, NO2, NOx, NH3, CO, SO2, O3, Benzene, Toluene, AQI, AQI_Bucket, Year_bin, Quarterly_bin]
Index: []
        Date Month_bin
1 2020-03-12     March
2 2020-03-13     March
3 2020-03-14     March
4 2020-03-15     March
5 2020-03-16     March


OBSERVING THE DATASET

In [16]:
df.head(40)

,City,Date,PM2.5,PM10,NO,NO2,NOx,NH3,CO,SO2,O3,Benzene,Toluene,AQI,AQI_Bucket,Year_bin,Quarterly_bin,Month_bin
1,Aizawl,2020-03-12,31.21,38.66,7.20,1.27,10.65,25.63,0.56,4.22,2.81,0.01,0.08,52.0,Satisfactory,2020,Jan-Mar,March
2,Aizawl,2020-03-13,38.39,46.68,7.19,0.91,10.37,29.16,0.57,4.46,0.18,0.00,0.00,60.0,Satisfactory,2020,Jan-Mar,March
3,Aizawl,2020-03-14,43.23,50.83,7.14,1.07,10.48,28.95,0.57,4.53,0.41,0.00,0.00,62.0,Satisfactory,2020,Jan-Mar,March
4,Aizawl,2020-03-15,33.82,41.03,7.09,0.36,9.73,28.41,0.48,4.63,0.30,0.00,0.00,70.0,Satisfactory,2020,Jan-Mar,March
5,Aizawl,2020-03-16,27.14,35.04,5.63,2.32,8.09,23.98,0.50,4.71,13.02,0.13,0.68,54.0,Satisfactory,2020,Jan-Mar,March
6,Aizawl,2020-03-17,27.32,35.75,3.07,2.14,3.41,24.57,0.48,4.84,6.03,0.25,1.34,40.0,Good,2020,Jan-Mar,March
7,Aizawl,2020-03-18,31.76,41.51,3.00,1.48,5.24,23.42,0.47,5.04,8.76,0.24,1.19,51.0,Satisfactory,2020,Jan-Mar,March
8,Aizawl,2020-03-19,43.80,53.59,2.97,1.31,4.97,23.41,0.48,5.30,9.96,0.26,1.11,63.0,Satisfactory,2020,Jan-Mar,March
9,Aizawl,2020-03-20,35.48,44.02,3.01,0.83,4.64,24.85,0.49,5.32,6.43,0.27,1.15,69.0,Satisfactory,2020,Jan-Mar,March
10,Aizawl,2020-03-21,51.27,60.05,3.01,0.88,4.62,24.44,0.49,5.63,8.62,49.66,50.11,70.0,Satisfactory,2020,Jan-Mar,March


TESTING TO SEE IF ANY MISSING VALUES IS REMAINING ON THE DATASET

In [17]:
print("\nMissing values")
print(df.isna().sum())


Missing values
City             0
Date             0
PM2.5            0
PM10             0
NO               0
NO2              0
NOx              0
NH3              0
CO               0
SO2              0
O3               0
Benzene          0
Toluene          0
AQI              0
AQI_Bucket       0
Year_bin         0
Quarterly_bin    0
Month_bin        0
dtype: int64


SAVING THE NEW DATASET AS PREPROCESSED AND MERGED DATA